# 5.5 · 支持向量机 / Support Vector Machine (SVM)

> **课程定位 / Where this fits**
> 第 5 课，**Part 5 · 监督学习：分类**。
> Lesson 5, **Part 5 · Supervised Classification**.
>
> 逻辑回归(5.1)随便找一条能分开的边界就行，但能分开的边界有**无数条**——哪条最好？SVM 给出一个漂亮的答案：**间隔(margin)最大**那条，它对扰动最不敏感、泛化最稳。再加**核技巧(kernel trick)**，SVM 还能画非线性边界。SVR(4.9)是它的回归版。深度学习兴起前，SVM 是分类的王者。
> Logistic regression (5.1) is happy with any separating boundary, but there are **infinitely many** — which is best? SVM answers elegantly: the one with the **largest margin**, most robust to perturbation and best for generalization. With the **kernel trick** it also draws nonlinear boundaries. SVR (4.9) is its regression cousin. Before deep learning, SVM was the champion of classification.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{w}, b$ —— 超平面 $\mathbf{w}^\top\mathbf{x}+b=0$ 的法向量与偏置 / weight vector & bias of the hyperplane
> - 间隔 margin $= 1/\|\mathbf{w}\|$ —— 边界到最近点的距离 / distance from boundary to nearest point
> - $C$ —— 软间隔正则系数（容忍违反的程度）/ soft-margin regularization (tolerance for violations)
> - $\gamma$ —— RBF 核的尺度 / RBF kernel scale
> - $K(\mathbf{x}_i,\mathbf{x}_j)$ —— 核函数 / kernel function

> 💡 **面试相关 / Interview-relevant**
> - "什么是间隔 / 为什么最大间隔泛化好"（★★★★★）
> - "支持向量是什么"（★★★★★）
> - "核技巧的本质 / 为什么不显式升维"（★★★★★）
> - "C 和 gamma 各控制什么"（★★★★★）
> - "hinge loss vs 逻辑损失" / "SVM 为什么要缩放"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解最大间隔的几何意义与支持向量。
   Understand the geometry of the maximum margin and support vectors.
2. 理解软间隔与 **C**（正则强度）。
   Understand the soft margin and **C**.
3. 从 **hinge loss** 视角看 SVM（对比逻辑损失）。
   View SVM through the **hinge loss** (vs logistic loss).
4. 理解**核技巧**：RBF/多项式核如何隐式升维。
   Understand the **kernel trick**: how RBF/polynomial kernels implicitly lift dimensions.
5. 调 **C 与 gamma** + 理解缩放必要性。
   Tune **C and gamma** and know why scaling is needed.

## 目录 / TOC
1. [先建直觉：哪条边界最好](#1)
2. [最大间隔与支持向量 ⭐](#2)
3. [软间隔与 C ⭐](#3)
4. [hinge loss ⭐](#4)
5. [核技巧 ⭐](#5)
6. [🔢 数据 Digits + C/gamma 调参](#6)
7. [缩放与多分类](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：哪条边界最好 / Intuition First

想象平面上两类点能被一条直线分开。能分开它们的直线有无穷多条——紧贴红点画一条也行，紧贴蓝点画一条也行。直觉上，**最好的那条应该"两边都留出最宽的空地"**，因为这样万一来个新点稍有偏差，也不容易越界判错。
Picture two classes separable by a line. Infinitely many lines work — one hugging the red points, one hugging the blue. Intuitively, **the best line leaves the widest empty strip on both sides**, because then a slightly-off new point is less likely to cross over and be misclassified.

这条"两边空地最宽"的线就是 **最大间隔(maximum margin)** 边界，正是 SVM 要找的。决定这条线位置的，只是**贴着空地边缘的那几个点**——它们叫**支持向量**，其余点挪一挪都不影响。
That widest-strip line is the **maximum-margin** boundary, exactly what SVM seeks. Its position is fixed by only **the few points touching the edge of the strip** — the **support vectors**; moving any other point doesn't change it.


<a id="2"></a>
## 2. 最大间隔与支持向量 ⭐ / Max Margin & Support Vectors

线性可分时，分隔超平面记作 $\mathbf{w}^\top\mathbf{x}+b=0$。可以推出**间隔**（边界到最近点的距离）等于 $\frac{1}{\|\mathbf{w}\|}$。要让间隔最大，就等价于让 $\|\mathbf{w}\|^2$ 最小：
For linearly separable data the hyperplane is $\mathbf{w}^\top\mathbf{x}+b=0$. One can show the **margin** (distance from boundary to nearest point) equals $\frac{1}{\|\mathbf{w}\|}$. Maximizing the margin is therefore equivalent to minimizing $\|\mathbf{w}\|^2$:

$$\min_{\mathbf{w},b}\ \tfrac12\|\mathbf{w}\|^2 \quad \text{s.t.}\quad y_i(\mathbf{w}^\top\mathbf{x}_i+b)\ge 1\ \ \forall i$$

约束 $y_i(\mathbf{w}^\top\mathbf{x}_i+b)\ge 1$ 的意思是"每个点都被正确分类，且落在间隔之外"。
The constraint $y_i(\mathbf{w}^\top\mathbf{x}_i+b)\ge 1$ says "every point is classified correctly and lies outside the margin".

**支持向量** = 恰好落在间隔边界上（约束取等号）的点。**只有它们决定边界**，其余点远离边界、移动不影响结果——这就是 SVM 的稀疏性与稳健性来源。
**Support vectors** = points lying exactly on the margin edge (constraint with equality). **Only they determine the boundary**; other points are far away and don't matter — the source of SVM's sparsity and robustness.

**为什么大间隔泛化好**：间隔大意味着边界离两类都远，对数据的小扰动不敏感；数学上 $\|\mathbf{w}\|^2$ 最小正是一种复杂度控制（类似 4.4 岭回归的正则）。
**Why large margin generalizes well:** a big margin keeps the boundary far from both classes, insensitive to small perturbations; mathematically minimizing $\|\mathbf{w}\|^2$ is a form of complexity control (like ridge regularization, 4.4).


<a id="3"></a>
## 3. 软间隔与 C ⭐ / Soft Margin & C

真实数据常常**线性不可分**或有噪声，硬性要求"所有点都在间隔外"会无解。**软间隔**引入松弛变量 $\xi_i\ge0$ 允许少量违反，并对违反量加惩罚：
Real data is often **not linearly separable** or noisy, so demanding "all points outside the margin" may be infeasible. The **soft margin** adds slack $\xi_i\ge0$ to allow some violations and penalizes them:

$$\min\ \tfrac12\|\mathbf{w}\|^2 + C\sum_i \xi_i \quad \text{s.t.}\quad y_i(\mathbf{w}^\top\mathbf{x}_i+b)\ge 1-\xi_i$$

**C 是正则旋钮**（方向与 4.4 岭回归的 $\lambda$ 相反）：
**C is the regularization knob** (opposite direction to ridge's $\lambda$, 4.4):
- **C 大 / large C**：重罚违反 → 间隔窄、努力分对每个点 → **低偏差、高方差**（过拟合）。
  Heavily penalizes violations → narrow margin, tries to get every point right → low bias, high variance (overfit).
- **C 小 / small C**：容忍违反 → 间隔宽、更平滑 → **高偏差、低方差**。
  Tolerates violations → wide margin, smoother → high bias, low variance.

可以粗略理解为 $C\approx 1/\lambda$。它是 SVM 最重要的超参之一。
Roughly $C\approx 1/\lambda$. One of SVM's most important hyperparameters.


<a id="4"></a>
## 4. hinge loss ⭐ / Hinge Loss

软间隔可以改写成"损失 + 正则"的形式（像 Part 4 所有模型那样）：
The soft margin can be rewritten as "loss + regularization" (like every model in Part 4):

$$\min_{\mathbf{w}} \;\underbrace{\sum_i \max(0,\,1 - y_i\,f(\mathbf{x}_i))}_{\text{hinge loss}} + \frac{1}{2C}\|\mathbf{w}\|^2$$

**hinge loss** $\max(0, 1-yf)$ 的含义：当点被正确分类且在间隔之外（$yf\ge1$）时，损失正好是 **0**；否则按违反程度线性惩罚。
The **hinge loss** $\max(0, 1-yf)$: when a point is correctly classified and outside the margin ($yf\ge1$), the loss is exactly **0**; otherwise it grows linearly with the violation.

对比逻辑损失：hinge 在 $yf>1$ 之后**完全为 0**（所以远离边界的点贡献为 0，产生**稀疏的支持向量**）；逻辑损失则永远 $>0$（所有点都参与）。
Versus logistic loss: hinge is **exactly 0** past $yf>1$ (points far from the boundary contribute nothing, giving **sparse support vectors**); logistic loss is always $>0$ (every point participates).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

z = np.linspace(-2, 3, 200)            # z = y·f(x), 同时表示"分类是否正确"和"置信度"
hinge = np.maximum(0, 1 - z)           # hinge: 当 z≥1 时为 0, 否则 = 1-z
logistic = np.log2(1 + np.exp(-z))     # 逻辑损失(以2为底), 永远>0
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z, hinge, lw=2, label="hinge (SVM): max(0,1-z)")
ax.plot(z, logistic, lw=2, label="logistic: log(1+e⁻ᶻ)")
ax.axvline(1, color="gray", ls="--", label="间隔边界 margin z=1")
ax.set_xlabel("z = y·f(x) (正确性×置信度 correctness×confidence)"); ax.set_ylabel("loss"); ax.legend()
ax.set_title("hinge 在 z≥1 后归零 → 稀疏支持向量; 逻辑损失永远>0\nhinge is 0 past z≥1 → sparse SVs; logistic always >0")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 核技巧 ⭐ / The Kernel Trick

线性不可分的数据，升到更高维往往就**可分**了（比如同心圆，加一个"到原点距离平方"的维度，内圈外圈就能用一个平面分开）。但显式地把每个点映射到高维 $\phi(\mathbf{x})$ 计算量极大。
Data that's not linearly separable often **becomes** separable in higher dimensions (e.g. concentric circles: add a "squared distance to origin" dimension and a plane separates inner from outer). But explicitly mapping each point to high-dim $\phi(\mathbf{x})$ is expensive.

**核技巧的精髓**：在 SVM 的对偶形式里，数据只以**内积** $\mathbf{x}_i^\top\mathbf{x}_j$ 的形式出现。我们用一个**核函数** $K(\mathbf{x}_i,\mathbf{x}_j)=\phi(\mathbf{x}_i)^\top\phi(\mathbf{x}_j)$ 直接算出"高维空间里的内积"，**而完全不需要真的算出 $\phi$**。常用核：
**The essence:** in SVM's dual form, data appears only through **inner products** $\mathbf{x}_i^\top\mathbf{x}_j$. We use a **kernel** $K(\mathbf{x}_i,\mathbf{x}_j)=\phi(\mathbf{x}_i)^\top\phi(\mathbf{x}_j)$ to compute the high-dim inner product directly, **without ever forming $\phi$**. Common kernels:
- **线性 linear**：$\mathbf{x}_i^\top\mathbf{x}_j$
- **多项式 polynomial**：$(\gamma\,\mathbf{x}_i^\top\mathbf{x}_j + r)^p$
- **RBF（高斯）**：$\exp(-\gamma\|\mathbf{x}_i-\mathbf{x}_j\|^2)$ —— 对应**无限维**空间，最常用。

**gamma($\gamma$)**（RBF 专属）控制单个样本的影响范围：**gamma 大** → 影响范围小、边界弯曲 → 过拟合；**gamma 小** → 影响范围大、边界平滑。
**gamma** (RBF) controls a single point's reach: large gamma → small reach, wiggly boundary → overfit; small gamma → large reach, smooth boundary.


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_circles
Xc, yc = make_circles(200, factor=0.4, noise=0.1, random_state=0)  # 同心圆: 线性不可分

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
# 网格点用于给整个平面上色, 显示决策区域 / grid to color the whole plane
xx, yy = np.meshgrid(np.linspace(-1.5,1.5,300), np.linspace(-1.5,1.5,300))
for ax, kern in zip(axes, ["linear", "rbf"]):
    m = SVC(kernel=kern, C=1, gamma="scale").fit(Xc, yc)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)   # 预测每个网格点的类别
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")               # 背景色=决策区域
    ax.scatter(Xc[:,0], Xc[:,1], c=yc, cmap="coolwarm", edgecolor="k", s=25)
    ax.set_title(f"kernel={kern}: 准确率 accuracy {m.score(Xc,yc):.2f}")
plt.tight_layout(); plt.show()
print("同心圆: 线性核无能为力, RBF 核轻松画圆形边界(隐式升维) / linear fails, RBF works")


<a id="6"></a>
## 6. 数据 Digits + C/gamma 调参 / Digits & Tuning

**Digits**：sklearn 内置手写数字（MNIST 的迷你版），1797 张 8×8 灰度图，0–9 十类。SVM 在这种图像分类上经典强。
**Digits**: sklearn's built-in handwritten digits (a mini MNIST), 1797 8×8 grayscale images, classes 0–9. SVM is classically strong on such image classification.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC

digits = load_digits()
print(f"Digits: {digits.data.shape}, 10 类 classes 0-9")
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, lab in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(str(lab)); ax.axis("off")
plt.suptitle("Digits 样例 samples (8×8)"); plt.tight_layout(); plt.show()

X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.3,
                                          stratify=digits.target, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)


In [ ]:
# 用网格搜索同时调 C 和 gamma / grid search over C and gamma
# GridSearchCV 对每个(C,gamma)组合做 3 折交叉验证, 选 CV 分数最高的
grid = GridSearchCV(SVC(kernel="rbf"),
                    {"C": [0.1, 1, 10], "gamma": [0.001, 0.01, 0.1]},
                    cv=3, n_jobs=-1).fit(Xtr, y_tr)
print(f"最佳参数 best params: {grid.best_params_}")
print(f"test 准确率 accuracy: {grid.score(Xte, y_te):.3f}")
# n_support_ 是每个类别的支持向量个数, 求和=总支持向量数
print(f"支持向量数 #support vectors: {grid.best_estimator_.n_support_.sum()} / {len(Xtr)} 训练样本")
print("只有部分样本成为支持向量 → SVM 的稀疏性 / only some samples are SVs → sparsity")


<a id="7"></a>
## 7. 缩放与多分类 / Scaling & Multiclass

- **必须缩放 / Must scale**：RBF 核含 $\|\mathbf{x}_i-\mathbf{x}_j\|^2$，和 KNN 一样对量纲敏感（3.4）。
  The RBF kernel contains $\|\mathbf{x}_i-\mathbf{x}_j\|^2$, so it's scale-sensitive like KNN (3.4).
- **多分类 / Multiclass**：SVM 本质是二分类。sklearn 的 `SVC` 用 **OvO**（one-vs-one，训练 $\binom{K}{2}$ 个两两分类器），`LinearSVC` 用 OvR。
  SVM is intrinsically binary. sklearn's `SVC` uses **OvO** ($\binom{K}{2}$ pairwise classifiers); `LinearSVC` uses OvR.
- **缺点 / Drawbacks**：大数据慢（训练约 $O(n^2)\sim O(n^3)$），且不直接给概率（需额外校准，见 5.15）。
  Slow on big data ($O(n^2)$–$O(n^3)$ training) and gives no direct probabilities (needs calibration, 5.15).


In [ ]:
# 缩放对 SVM 的影响 / scaling impact
# Digits 像素本就在同一 0-16 量纲, 缩放帮助不大; 故意把一个特征×1000 暴露问题
Xb_tr = X_tr.copy(); Xb_tr[:, 0] *= 1000
Xb_te = X_te.copy(); Xb_te[:, 0] *= 1000
unscaled = SVC(kernel="rbf", gamma="scale").fit(Xb_tr, y_tr).score(Xb_te, y_te)
scaler = StandardScaler().fit(Xb_tr)          # 在(被破坏的)训练集上拟合缩放器
scaled = SVC(kernel="rbf", gamma="scale").fit(scaler.transform(Xb_tr), y_tr).score(scaler.transform(Xb_te), y_te)
print(f"某特征×1000 不缩放 unscaled: {unscaled:.3f}   标准化后 scaled: {scaled:.3f}")
print("(同量纲缩放影响小, 但量纲不一时不缩放会被大特征绑架, 同 KNN)")

# 多分类: SVC 内部训练 C(10,2)=45 个 OvO 两两分类器 / SVC trains 45 pairwise classifiers
svc = SVC(decision_function_shape="ovo").fit(Xtr, y_tr)
print(f"\nSVC 用 OvO; decision_function(ovo) 形状 shape = {svc.decision_function(Xte).shape}")
print("(45 = C(10,2), 10 类两两配对; 默认 'ovr' 会把它们聚合成 10 列 / default ovr aggregates to 10)")


<a id="8"></a>
## 8. 小结 / Summary

```
SVM: 最大间隔 = 1/‖w‖ 最大化; 只有支持向量(间隔边界上的点)决定边界
软间隔 + C: C 大→窄间隔/过拟合, C 小→宽间隔/欠拟合 (C≈1/λ)
hinge loss: max(0,1-yf), z≥1 归零 → 稀疏支持向量 (vs 逻辑损失永远>0)
核技巧: 数据只以内积出现 → 用 K(xᵢ,xⱼ) 算高维内积, 不显式升维
  RBF: exp(-γ‖xᵢ-xⱼ‖²), γ大→边界弯曲/过拟合
必须缩放; 多分类 SVC=OvO; 大数据慢, 不直接给概率
```

### 💡 面试速查 / Interview cheat-sheet
1. **最大间隔**最稳健；**支持向量**是唯一决定边界的点。
   Max margin is most robust; support vectors alone fix the boundary.
2. **C** 控正则（大=过拟合）；**gamma** 控 RBF 影响范围（大=过拟合）。
   C controls regularization (large=overfit); gamma controls RBF reach (large=overfit).
3. **核技巧**：用核函数算高维内积，避免显式升维；RBF 对应无限维。
   Kernel trick computes high-dim inner products via a kernel, avoiding explicit lifting; RBF = infinite-dim.
4. **hinge loss** 产生稀疏支持向量；**必须缩放**。
   Hinge loss yields sparse support vectors; scaling is mandatory.
5. 大数据慢、不直接给概率是 SVM 的主要缺点。
   Slow on big data and no direct probabilities are SVM's main drawbacks.

### 下一节 / Next
**5.6 决策树**——SVM 是平滑边界，决策树是**轴对齐的阶梯**边界，可解释性极强，也是随机森林/GBDT 的基石。
**5.6 Decision Tree** — SVM gives smooth boundaries; trees give **axis-aligned, staircase** boundaries, highly interpretable, and the building block of Random Forest/GBDT.
